# Faster R-CNN 평가 (구 테스트 8장, 새 테스트 22장)

저장된 가중치로 두 테스트셋을 평가하고, 결과를 `results/metrics_test_old8.json`, `results/metrics_test_new22.json`에 저장합니다.

**준비 (Colab)**
1. 런타임 → 런타임 유형 변경 → **T4 GPU** (추론 시간을 YOLO와 같은 조건에서 재기 위해)
2. 드라이브 `MyDrive/pcb-aoi/faster_rcnn/`에 `config.py`, `common.py`, 이 노트북, 가중치(`config.WEIGHTS_NAME`, 현재 `best_fasterrcnn_20260923_1007.pth`)가 있어야 합니다.
3. 결과 json은 드라이브 `pcb-aoi/faster_rcnn/results/`에 저장됩니다. 저장소의 `results/`로 옮겨 커밋하세요.

**클래스 id 처리**: 평가 전에 각 테스트셋 json의 id를 이름 기준으로 학습 라벨에 다시 매핑하고, 그 결과를 표로 출력합니다. 2026-09-23 평가에서는 두 테스트셋 모두 id와 이름이 학습 json과 같았습니다. 다른 테스트셋을 추가할 때 표 아래 문구로 확인하세요.

**커밋 전**: 5절(예측 시각화) 셀의 이미지 출력은 지우고 커밋하세요.

## 1. 환경 설정

In [ ]:
# Colab이면 드라이브를 연결하고 config.py·common.py가 있는 드라이브 폴더로 이동한다.
# 로컬이면 이 노트북과 같은 폴더(faster_rcnn/)의 파일을 그대로 쓴다.
import os, sys, subprocess, importlib

PROJECT_DIR = "/content/drive/MyDrive/pcb-aoi/faster_rcnn"   # config.py, common.py를 올려 둔 드라이브 폴더

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torchmetrics"], check=True)
    os.chdir(PROJECT_DIR)
    if PROJECT_DIR not in sys.path:
        sys.path.insert(0, PROJECT_DIR)

import config, common
importlib.reload(config); importlib.reload(common)   # config.py를 고친 뒤 이 셀만 다시 실행해도 반영됨

for k, v in config.summary().items():
    print(f"{k:13s} {v}")

In [ ]:
import torch, torchvision
print("torch:", torch.__version__, "/ torchvision:", torchvision.__version__)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU 없음: 런타임 → 런타임 유형 변경 → T4 GPU로 바꾸세요.")

## 2. 모델 불러오기

가중치는 학습 json의 category_id를 그대로 라벨로 학습했습니다. `config.CLASSES` 순서가 학습 json과 같은지 먼저 확인합니다.

In [ ]:
common.check_train_classes(config.DATA_ROOT / "train" / common.ANN_FILE, config.CLASSES)

print("가중치:", config.WEIGHTS)
model = common.load_model(config.WEIGHTS, config.NUM_CLASSES, device)
print("불러오기 완료")

## 3. 두 테스트셋 평가

- mAP50, mAP50-95, mAR100: torchmetrics (모든 예측 사용)
- 혼동행렬, P/R/F1, 분류 정확도, 검출률: IoU ≥ 0.5, 신뢰도 ≥ 0.5로 매칭 (기존 노트북과 같은 방식)
- 추론 시간: 첫 이미지로 워밍업 5회 후 50회 측정

In [ ]:
import datetime

TEST_SETS = {
    "old8":  {"dir": config.OLD_TEST_DIR, "id_fix": {},                     "expected": 8},
    "new22": {"dir": config.NEW_TEST_DIR, "id_fix": config.NEW_TEST_ID_FIX, "expected": 22},
}

results, datasets = {}, {}
for key, s in TEST_SETS.items():
    print(f"\n{'#' * 70}\n# {key}: {s['dir']}\n{'#' * 70}")
    ds = common.CocoDetectionDataset(s["dir"], config.CLASSES, id_fix=s["id_fix"])
    if len(ds) != s["expected"]:
        print(f"주의: 이미지가 {len(ds)}장입니다 (예상 {s['expected']}장). 경로를 확인하세요.")

    res = common.evaluate(model, ds, device, config.CLASSES, config.IOU_TH, config.SCORE_TH)
    res["latency"] = common.measure_latency(model, ds, device,
                                            config.LATENCY_WARMUP, config.LATENCY_RUNS)
    res["meta"] = {
        "test_set": key,
        "test_dir": str(s["dir"]),
        "weights": config.WEIGHTS_NAME,
        "model": config.MODEL_NAME,
        "label_mapping": [{"json_id": cid, "json_name": raw, "class": name, "note": note}
                          for cid, raw, name, note in ds.mapping_rows],
        "device": torch.cuda.get_device_name(0) if device.type == "cuda" else "cpu",
        "torch": torch.__version__,
        "evaluated_at": datetime.datetime.now().isoformat(timespec="seconds"),
    }
    results[key], datasets[key] = res, ds

    print()
    common.print_report(res, key)
    common.save_json(res, config.RESULTS_DIR / f"metrics_test_{key}.json")

## 4. 비교 표와 혼동행렬

README 결과 표에 그대로 옮길 수 있는 형태로 출력합니다.

In [ ]:
ROWS = [
    ("mAP50",              lambda r: r["map50"]),
    ("mAP50-95",           lambda r: r["map50_95"]),
    ("mAR100",             lambda r: r["mar100"]),
    ("Precision (macro)",  lambda r: r["macro"]["precision"]),
    ("Recall (macro)",     lambda r: r["macro"]["recall"]),
    ("F1 (macro)",         lambda r: r["macro"]["f1"]),
    ("분류 정확도",         lambda r: r["accuracy"]),
    ("검출률",              lambda r: r["detection_rate"]),
    ("추론 평균 (ms)",      lambda r: r["latency"]["mean_ms"]),
    ("추론 최대 (ms)",      lambda r: r["latency"]["max_ms"]),
]
keys = list(results)
print("| 지표 | " + " | ".join(f"{k} ({results[k]['images']}장)" for k in keys) + " |")
print("|---|" + "---|" * len(keys))
for name, get in ROWS:
    print(f"| {name} | " + " | ".join(f"{get(results[k]):.3f}" if "ms" not in name
                                      else f"{get(results[k]):.1f}" for k in keys) + " |")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Colab 기본 폰트는 한글이 깨지므로 그림에는 영어만 쓴다.
fig, axes = plt.subplots(1, len(results), figsize=(9 * len(results), 8))
axes = np.atleast_1d(axes)
for ax, (key, res) in zip(axes, results.items()):
    cm = np.array(res["confusion_matrix"]["rows_gt_cols_pred"])
    labels = res["confusion_matrix"]["labels"]
    ax.imshow(cm, cmap="Blues")
    ax.set_xticks(range(len(labels)))
    ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Ground truth")
    ax.set_title(f"{key}  (IoU>={config.IOU_TH}, conf>={config.SCORE_TH})")
    for r in range(len(labels)):
        for c in range(len(labels)):
            if cm[r, c]:
                ax.text(c, r, cm[r, c], ha="center", va="center",
                        color="white" if cm[r, c] > cm.max() * 0.5 else "black")
plt.tight_layout()
plt.show()

## 5. 예측 시각화 (선택)

정답 박스(빨간 점선)와 예측 박스(초록)를 겹쳐 그립니다. 빨간 박스만 있고 초록 박스가 없으면 미검출입니다.
**이 셀의 이미지 출력은 커밋 전에 지우세요** (노트북 용량이 크게 늘어남).

In [ ]:
import matplotlib.patches as patches

VIS_SET = "new22"   # "old8" 또는 "new22"
N_SHOW = 4

ds = datasets[VIS_SET]
n = min(N_SHOW, len(ds))
cols = 2
rows_ = (n + cols - 1) // cols
fig, axes = plt.subplots(rows_, cols, figsize=(7 * cols, 6 * rows_))
axes = np.atleast_1d(axes).ravel()

model.eval()
for ax, idx in zip(axes, range(n)):
    img, target = ds[idx]
    with torch.no_grad():
        pred = {k: v.cpu() for k, v in model([img.to(device)])[0].items()}
    ax.imshow(img.permute(1, 2, 0))
    for box in target["boxes"]:
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       linewidth=1.5, edgecolor="red", linestyle="--"))
    for box, label, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
        if score < config.SCORE_TH:
            continue
        x1, y1, x2, y2 = box.tolist()
        ax.add_patch(patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False,
                                       linewidth=2, edgecolor="lime"))
        ax.text(x1, y1 - 3, f"{config.CLASSES[int(label) - 1]} {score:.2f}",
                color="lime", fontsize=8)
    ax.set_title(f"{VIS_SET}[{idx}]  GT {len(target['boxes'])}")
    ax.axis("off")
for ax in axes[n:]:
    ax.axis("off")
plt.tight_layout()
plt.show()

## 6. requirements.txt 기록

평가한 환경의 실제 패키지 버전을 `requirements.txt`로 남깁니다 (train.ipynb 6절과 같은 내용).

In [ ]:
common.write_requirements(config.HERE / "requirements.txt")